## Association Rules Analysis

In [ ]:
!pip install mlxtend

In [ ]:
import mlxtend

mlxtend.__version__

In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.__version__

## Read the dataset

In [ ]:
bb_df = pd.read_csv("https://raw.githubusercontent.com/manaranjanp/MLUL2/refs/heads/main/arules/bb.csv")
bb_df.head()

In [ ]:
bb_df.info()

## Exploring the dataset

How many unique orders and SKUs?

In [ ]:
len(bb_df.Order.unique())

In [ ]:
len(bb_df.SKU.unique())

In [ ]:
sku_desc_dict = dict(zip(bb_df.SKU, bb_df.Description))

## Storing the SKU number and their description in a dictionary

In [ ]:
len(sku_desc_dict)

In [ ]:
# Print few SKU's and their description
for k, v in random.sample(list(sku_desc_dict.items()), 5):
    print(f"{k}: {v}")

## Most Sold Items

In [ ]:
most_sold_df = bb_df.SKU.value_counts().reset_index()

In [ ]:
most_sold_df.columns = ['SKU', 'Count']

In [ ]:
items_freq_df = most_sold_df.sort_values("Count",
                                      ascending = False)

In [ ]:
items_freq_df['Description'] = items_freq_df.SKU.map(lambda x:sku_desc_dict[x])

In [ ]:
items_freq_df[0:10]

In [ ]:
items_freq_df[-10:]

In [ ]:
low_freq_skus = list(items_freq_df[items_freq_df.Count < 3]['SKU'])

In [ ]:
len(low_freq_skus)

In [ ]:
plt.figure(figsize=(15,4))
plt.hist(items_freq_df['Count'], bins = range(0, 1500, 50));
plt.xticks(range(0, 1500, 50));

## Encoding the Baskets


In [ ]:
bb_df.drop( 'Description', axis = 1, inplace = True)

In [ ]:
bb_df['Quantity'] = 1

In [ ]:
bb_df.head()

In [ ]:
bb_df.info()

In [ ]:
bb_df['Quantity'] = bb_df['Quantity']

In [ ]:
bb_pivot_df = bb_df.pivot( index='Order',
                          columns='SKU',
                          values = "Quantity" ).astype('float16').reset_index(drop=True)

In [ ]:
bb_pivot_df.head(5)

In [ ]:
bb_pivot_df.fillna( 0, inplace = True )

In [ ]:
bb_pivot_df.head(5)

In [ ]:
bb_pivot_df.info()

In [ ]:
bb_pivot_df.shape

## Apriori : Finding Itemsets


In [ ]:
frequent_itemsets = apriori(bb_pivot_df,
                            min_support=0.002,
                            use_colnames=True,
                            low_memory = True)

In [ ]:
frequent_itemsets.tail(5)

In [ ]:
plt.hist(frequent_itemsets.support, bins = 20);

## Finding Association Rules

In [ ]:
association_rules?

In [ ]:
rules = association_rules(frequent_itemsets,
                          metric="lift",
                          min_threshold=1,
                          return_metrics = ['antecedent support',
                                            'consequent support',
                                            'support',
                                            'confidence',
                                            'lift',
                                            'conviction'])
rules.head(10)

In [ ]:
rules.info()

## Making sense of the rules

In [ ]:
def get_sku_description( skus ):
    all_skus = []
    for sku in skus:
         all_skus.append(sku_desc_dict[sku])
    return ",".join(all_skus)

In [ ]:
rules.antecedents[0]

In [ ]:
rules['antecedents_desc'] = rules.antecedents.map(lambda rec: sku_desc_dict[list(rec)[0]])

In [ ]:
rules['consequents_desc'] = rules.consequents.map(lambda rec: sku_desc_dict[list(rec)[0]])

In [ ]:
rules = rules.sort_values('confidence', ascending=False)

In [ ]:
rules

In [ ]:
rules.to_excel("rules.xlsx", index = False)

In [ ]:
top_10_rules = rules.sort_values('confidence', ascending=False)[0:10]

In [ ]:
rules_crosstab = pd.crosstab(rules.consequents_desc,
                             rules.antecedents_desc,
                             values = rules.lift,
                             aggfunc = 'max')

In [ ]:
rules_crosstab

In [ ]:
sn.heatmap(rules_crosstab, cmap = 'Blues', vmin=0, vmax=1);